# Practice 106 — Double ML & Heterogeneous Treatment Effects

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_dataset
from src.plotting import bias_comparison_plot, cate_calibration_scatter, policy_value_curve_plot

## Phase 1 — The naive ML plug-in estimator

We simulate data with a **known** heterogeneous CATE `tau(x)` and a known average
treatment effect (see `src/datasets.py`). Treatment is confounded with the outcome
through `x0`, which is what makes the naive estimator's bias show up rather than
being invisible noise.

In [ ]:
data = load_dataset(n=1000, seed=0)
print(f"True ATE: {data.ate_true:.3f}")
data.as_frame().head()

### Exercise — `src/_01_naive_bias.py :: naive_plugin_theta`

Open `src/_01_naive_bias.py`, read the `TODO(human)` block above the function,
implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_naive_bias import default_model_y, naive_plugin_theta

naive_theta = naive_plugin_theta(data.X, data.D, data.Y, default_model_y())
print(f"Naive plug-in theta_hat: {naive_theta:.3f}  (bias = {naive_theta - data.ate_true:+.3f})")

## Phase 2 — Cross-fitting

Same nuisance models as Phase 1 (a gradient-boosted regressor for `E[Y|X]`, a
gradient-boosted classifier for the propensity `E[D|X]`), but now fit via K-fold
sample splitting so no observation's residual comes from a model that saw that
observation during training.

### Exercise — `src/_02_cross_fitting.py :: cross_fit_residuals`

Open `src/_02_cross_fitting.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._02_cross_fitting import cross_fit_residuals

Y_resid, D_resid = cross_fit_residuals(data.X, data.D, data.Y)
print(f"Y_resid std: {Y_resid.std():.3f}   D_resid std: {D_resid.std():.3f}")

## Phase 3 — The partially linear DML moment, and the headline comparison

The Neyman-orthogonal moment condition, solved as a residual-on-residual regression
(Robinson, 1988). Combined with Phase 1's naive estimator and Phase 2's cross-fitting,
we now have all three estimators the practice compares: **naive**, **orthogonal without
cross-fitting**, and **full DML**.

### Exercise — `src/_03_partially_linear_dml.py :: dml_theta`

Open `src/_03_partially_linear_dml.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._03_partially_linear_dml import dml_theta, dml_without_crossfit, full_dml_theta

theta_dml = dml_theta(Y_resid, D_resid)
theta_no_cf = dml_without_crossfit(data.X, data.D, data.Y)
theta_full = full_dml_theta(data.X, data.D, data.Y)
print(f"True ATE:              {data.ate_true:.3f}")
print(f"Naive:                 {naive_theta:.3f}")
print(f"DML (residuals above): {theta_dml:.3f}")
print(f"DML, no cross-fit:     {theta_no_cf:.3f}")
print(f"Full DML:              {theta_full:.3f}")

### The headline demonstration: bias over many replications

One draw can be lucky. We redraw the dataset 200 times, refit all three estimators
on every draw, and look at the *distribution* of `theta_hat - true_ATE` each one
produces — the bias orthogonality plus cross-fitting removes should be visible here,
not asserted.

In [ ]:
from src._03_partially_linear_dml import bias_monte_carlo

biases = bias_monte_carlo(n_reps=200, n=500, seed=1)
for name, b in biases.items():
    print(f"{name:16s} mean bias = {b.mean():+.3f}   std = {b.std():.3f}")

fig = bias_comparison_plot(biases)
fig

### Optional cross-check: EconML's `LinearDML`

`EconML` implements the same partially linear DML model (Chernozhukov et al., 2018)
behind a higher-level API. This cell is a sanity check, not new teaching content — if
`econml` didn't install cleanly in your environment, skip it, the practice does not
depend on it.

In [ ]:
try:
    from econml.dml import LinearDML
    from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

    est = LinearDML(
        model_y=GradientBoostingRegressor(n_estimators=50, max_depth=2, random_state=0),
        model_t=GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=0),
        discrete_treatment=True,
        cv=5,
        random_state=0,
    )
    est.fit(data.Y, data.D, X=data.X)
    print(f"EconML LinearDML ATE:  {est.ate(data.X):.3f}   (ours, full DML: {theta_full:.3f})")
except ImportError as e:
    print(f"(econml not installed -- skipping cross-check: {e})")

## Phase 4 — Heterogeneous treatment effects: the T-learner

Phases 1-3 estimate one constant `theta` — the average effect. Real treatment effects
usually vary by unit; the T-learner fits one outcome model per arm and differences
their predictions to estimate `tau(x)`.

### Exercise — `src/_04_meta_learner.py :: t_learner_cate`

Open `src/_04_meta_learner.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._04_meta_learner import t_learner_cate

tau_hat = t_learner_cate(data.X, data.D, data.Y)
mae = np.mean(np.abs(tau_hat - data.tau_true))
print(f"T-learner CATE MAE vs. true tau(x): {mae:.3f}")

fig = cate_calibration_scatter(data.tau_true, tau_hat)
fig

### Optional cross-check: a causal forest

`EconML`'s `CausalForestDML` estimates CATE with an ensemble of honest, orthogonalized
trees instead of a single meta-learner split — the more automatic (and more expensive)
tool practitioners usually reach for first. Again optional: skip if `econml` is not
installed.

In [ ]:
try:
    from econml.dml import CausalForestDML

    cf = CausalForestDML(n_estimators=200, random_state=0, discrete_treatment=True, cv=5)
    cf.fit(data.Y, data.D, X=data.X)
    tau_hat_cf = cf.effect(data.X)
    mae_cf = np.mean(np.abs(tau_hat_cf - data.tau_true))
    print(f"Causal forest CATE MAE vs. true tau(x): {mae_cf:.3f}  (T-learner: {mae:.3f})")
except ImportError as e:
    print(f"(econml not installed -- skipping cross-check: {e})")

## Phase 5 — From CATE to policy

A CATE estimate only matters if it changes a decision. We turn `tau_hat` into a
budget-constrained treatment policy ("treat whoever benefits most, subject to a
budget") and evaluate its value via inverse-propensity weighting.

### Exercise — `src/_05_policy_value.py :: policy_value`

Open `src/_05_policy_value.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._05_policy_value import policy_value, policy_value_curve

budgets, cate_values = policy_value_curve(tau_hat, data)
_, random_values = policy_value_curve(np.random.default_rng(0).permutation(tau_hat), data, budgets)

fig = policy_value_curve_plot(budgets, {"CATE-targeted": cate_values, "random assignment": random_values})
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert abs(naive_theta - theta_full) > 0, "naive and full DML should generally disagree"
assert tau_hat.shape == data.tau_true.shape
assert cate_values.shape == budgets.shape
# Targeting treatment by CATE should be worth at least as much as random assignment
# at the same budget, on average across budgets.
assert cate_values.mean() >= random_values.mean() - 0.5, "CATE-targeted policy should not be much worse than random"
print("OK")